###Contextualização e Desafio

Este notebook implementa um pipeline de sanitização de dados para os datasets de produtos e pedidos da Olist, utilizando apenas bibliotecas nativas do Python (`csv`, `re`, `datetime`). O objetivo é tratar inconsistências como dados ausentes, padronizar strings e aplicar regras de negócio específicas para garantir a qualidade dos dados para relatórios e modelos de Machine Learning.

 ## 1. Configuração e Importação de Bibliotecas

Primeiro, vamos importar as bibliotecas necessárias para as operações de arquivo, expressões regulares e manipulação de datas.

In [14]:
import csv
import re
from datetime import datetime
#import requests # Para baixar os arquivos do GitHub
#import os # Para manipular arquivos

 ## 2. Download dos Datasets

Vamos baixar os arquivos `olist_products_dataset.csv` e `olist_orders_dataset.csv` diretamente do repositório do GitHub fornecido.

In [15]:
# def baixando_arquivo(url, local_arquivo):
#     """Baixa um arquivo de uma URL e salva localmente."""
#     print(f"Baixando {local_arquivo}...")
#     with requests.get(url, stream=True) as r:
#         r.raise_for_status()
#         with open(local_arquivo, 'wb') as f:
#             for chunk in r.iter_content(chunk_size=8192):
#                 f.write(chunk)
#     print(f"Download de {local_arquivo} concluído.")

# # URLs dos arquivos brutos no GitHub
# produtos_url = 'https://raw.githubusercontent.com/fiesc-junior-prado/mine_projeto_bloco_1/main/olist_products_dataset.csv'
# pedidos_url = 'https://raw.githubusercontent.com/fiesc-junior-prado/mine_projeto_bloco_1/main/olist_orders_dataset.csv'

# # Nomes dos arquivos locais
# arquivo_produtos = 'olist_products_dataset.csv'
# arquivo_pedidos = 'olist_orders_dataset.csv'

# # Executar download
# baixando_arquivo(produtos_url, arquivo_produtos)
# baixando_arquivo(pedidos_url, arquivo_pedidos)

In [16]:
with open("/content/drive/MyDrive/olist_orders_dataset.csv", "r", newline='', encoding="utf-8") as arquivo:
    leitor = csv.DictReader(arquivo)
    pedidos = list(leitor)

with open("/content/drive/MyDrive/olist_products_dataset.csv", "r", newline='', encoding="utf-8") as arquivo:
    leitor = csv.DictReader(arquivo)
    produtos = list(leitor)

arquivo_produtos = '/content/drive/MyDrive/olist_products_dataset.csv'
arquivo_pedidos = '/content/drive/MyDrive/olist_orders_dataset.csv'

 ## 3. Validação e Tratamento de Dados Ausentes, Padronização de Strings e Regex (Dataset de Produtos)

Esta etapa irá processar o `olist_products_dataset.csv` para:
- Preencher `product_category_name` nulo/vazio com "Sem Categoria".
- Tratar valores nulos nas dimensões físicas (`product_weight_g`, `product_length_cm`, `product_height_cm`, `product_width_cm`) atribuindo `0` (zero) como valor padrão, justificando que, para este cenário, um produto sem especificação de peso/dimensão pode ser tratado como um item de custo zero para cálculo de frete ou que será verificado manualmente. Isso evita descartar registros importantes sem uma política de negócio mais detalhada.
- Converter nomes de categorias para minúsculas e remover espaços em branco excedentes.
- Limpar caracteres especiais/pontuações indevidas usando Expressões Regulares.

In [17]:
def processar_produtos(caminho_arquivo):

    """Processa o dataset de produtos para tratar nulos, padronizar strings e limpar categorias.
    Retorna os dados processados, o total de linhas e a contagem de categorias corrigidas."""

    produtos_sanitizados = []
    total_produtos_processados = 0
    categorias_nulas_corrigidas = 0
    dimensoes_nulas_corrigidas = 0

    # Colunas de dimensões físicas que precisam de tratamento de nulos
    colunas_dimensoes_fisicas = [
        'product_weight_g', 'product_length_cm',
        'product_height_cm', 'product_width_cm'
    ]
    with open(caminho_arquivo, 'r', encoding='utf-8') as arquivo:
        leitor = csv.DictReader(arquivo)
        # Captura os nomes dos campos para manter a ordem e incluir novas colunas, se houver
        fieldnames = leitor.fieldnames
        if 'product_category_name_sanitized' not in fieldnames:
            fieldnames.append('product_category_name_sanitized')

        for row in leitor:
            total_produtos_processados += 1

            # --- 1. Validação e Tratamento de Dados Ausentes (product_category_name) ---
            categoria = row.get('product_category_name')
            if not categoria or categoria.strip() == '':
                row['product_category_name'] = 'Sem Categoria'
                categorias_nulas_corrigidas += 1

            # --- 1. Validação e Tratamento de Dados Ausentes (Dimensões Físicas) ---
            for coluna in colunas_dimensoes_fisicas:
                valor = row.get(coluna)
                if not valor or valor.strip() == '':
                    row[coluna] = '0' # Atribuir 0 como valor padrão
                    dimensoes_nulas_corrigidas += 1

            # --- 2. Padronização de Strings e Regex ---
            # Pega a categoria (que já pode ter sido corrigida de nula)
            categoria_limpar = row['product_category_name']

            # Converte para minúsculas e remove espaços
            categoria_limpar = categoria_limpar.lower().strip()

            # Remove caracteres especiais ou pontuações indevidas
            # Mantém apenas letras, números e espaços (que serão tratados se necessário)
            categoria_limpar = re.sub(r'[^a-z0-9\s]', '', categoria_limpar)

            # Remove múltiplos espaços internos e espaços nas extremidades novamente
            categoria_limpar = re.sub(r'\s+', ' ', categoria_limpar).strip()

            row['product_category_name_sanitized'] = categoria_limpar

            produtos_sanitizados.append(row)

    return produtos_sanitizados, total_produtos_processados, categorias_nulas_corrigidas, dimensoes_nulas_corrigidas

## 4. Lógica de Regra de Negócio e Formatação Temporal (olist_orders_dataset.csv)

Nesta seção, trataremos o arquivo `olist_orders_dataset.csv`:
1. **Regra de Negócio:** Validar a hipótese de que `order_delivered_customer_date` está nulo quando o `order_status` é 'canceled'.
2. **Formatação Temporal:** Converter `order_approved_at` para o formato de data simplificado brasileiro (DD/MM/YYYY).

In [18]:
def processar_pedidos(caminho_arquivo):
    """
    Processa o dataset de pedidos para tratar datas nulas, verificar status e formatar datas.
    Retorna os dados processados, o total de linhas, e contagens relacionadas a datas de entrega e cancelamentos.
    """
    pedidos_sanitizados = []
    total_pedidos_processados = 0
    pedidos_data_entrega_nula = 0
    pedidos_cancelados_com_data_entrega_nula = 0
    pedidos_cancelados_outros = 0 # Pedidos cancelados que podem ter data de entrega (raro, mas possível)

    with open(caminho_arquivo, mode='r', encoding='utf-8') as arquivo:
        leitor = csv.DictReader(arquivo)
        fieldnames = leitor.fieldnames
        # Adiciona nova coluna para a data formatada
        if 'order_approved_at_formatted' not in fieldnames:
            fieldnames.append('order_approved_at_formatted')

        for row in leitor:
            total_pedidos_processados += 1

            # --- 3. Lógica de Regra de Negócio (Filtros e Validação) ---
            data_entrega = row.get('order_delivered_customer_date')
            status_pedido = row.get('order_status')

            if not data_entrega or data_entrega.strip() == '':
                pedidos_data_entrega_nula += 1
                if status_pedido == 'canceled':
                    pedidos_cancelados_com_data_entrega_nula += 1
            elif status_pedido == 'canceled':
                # Captura pedidos cancelados que, por algum motivo, não têm data de entrega nula
                pedidos_cancelados_outros += 1

            # --- 4. Formatação Temporal (Datetime) ---
            aprovado = row.get('order_approved_at')
            if aprovado and aprovado.strip() != '':
                try:
                    # Tenta parsear a data e formatar
                    objeto_dt = datetime.strptime(aprovado, '%Y-%m-%d %H:%M:%S')
                    row['order_approved_at_formatted'] = objeto_dt.strftime('%d/%m/%Y')
                except ValueError:
                    row['order_approved_at_formatted'] = 'Data Inválida'
            else:
                row['order_approved_at_formatted'] = 'Data Ausente'

            pedidos_sanitizados.append(row)

    return (
        pedidos_sanitizados,
        total_pedidos_processados,
        pedidos_data_entrega_nula,
        pedidos_cancelados_com_data_entrega_nula,
        pedidos_cancelados_outros
    )

 ## 5. Relatório de Status Manual e Execução Principal

Esta seção executa as funções de processamento e gera um sumário estatístico final, conforme os requisitos do projeto.

In [19]:
# --- Execução do Pipeline ---

print("\n--- Iniciando o Pipeline de Sanitização de Dados ---")

# Processar produtos
produtos_processados_data, total_produtos_proc, cat_nulas_corr, dim_nulas_corr = processar_produtos(arquivo_produtos)

print(f"\nTotal de linhas de produtos processadas: {total_produtos_proc}")
print(f"Categorias de produtos nulas corrigidas para 'Sem Categoria': {cat_nulas_corr}")
print(f"Dimensões físicas de produtos nulas/vazias corrigidas para '0': {dim_nulas_corr}")

# Processar pedidos
pedidos_processados_data, total_pedidos_proc, data_entrega_nula, cancelados_com_data_nula, cancelados_outros = processar_pedidos(arquivo_pedidos)

print(f"\nTotal de linhas de pedidos processadas: {total_pedidos_proc}")
print(f"Pedidos com 'order_delivered_customer_date' nula/vazia: {data_entrega_nula}")
print(f"Pedidos cancelados com 'order_delivered_customer_date' nula/vazia: {cancelados_com_data_nula}")

# Verificar a hipótese de negócio:
# Todos os pedidos com data de entrega nula são cancelados?
# A hipótese é que 'data_entrega_nula' == 'cancelados_com_data_nula'

if data_entrega_nula == cancelados_com_data_nula:
    print("**Hipótese de Negócio Olist COMPROVADA:** Todos os pedidos com data de entrega nula têm o status 'canceled'.")
else:
    print("**Hipótese de Negócio Olist PARCIALMENTE COMPROVADA/NÃO COMPROVADA:** Nem todos os pedidos com data de entrega nula têm o status 'canceled'.")
    print(f"    Pedidos com data de entrega nula mas status diferente de 'canceled': {data_entrega_nula - cancelados_com_data_nula}")

print(f"Pedidos com status 'canceled' mas com 'order_delivered_customer_date' preenchida: {cancelados_outros}")

print("\n--- Sumário Estatístico Final ---")
print(f"Total de linhas processadas (Produtos): {total_produtos_proc}")
print(f"Total de linhas processadas (Pedidos): {total_pedidos_proc}")
print(f"Total de registros nulos corrigidos (Categorias de Produtos): {cat_nulas_corr}")
print(f"Total de registros nulos corrigidos (Dimensões Físicas de Produtos): {dim_nulas_corr}")
print(f"Total de pedidos cancelados identificados (com data de entrega nula): {cancelados_com_data_nula}")
print(f"Total de pedidos com data de aprovação formatada.")

# Exemplo de visualização de algumas linhas sanitizadas (opcional, para ver o resultado)
print("\n--- Exemplo de dados de produtos sanitizados (primeiras 5 linhas) ---")
for i, prod in enumerate(produtos_processados_data[:5]):
    print(f"Produto {i+1}: Categoria original: '{prod.get('product_category_name')}', Sanitizado: '{prod.get('product_category_name_sanitized')}', Peso: '{prod.get('product_weight_g')}'")

print("\n--- Exemplo de dados de pedidos sanitizados (primeiras 5 linhas) ---")
for i, ped in enumerate(pedidos_processados_data[:5]):
    print(f"Pedido {i+1}: Aprovado em: '{ped.get('order_approved_at')}', Formatado: '{ped.get('order_approved_at_formatted')}', Entrega: '{ped.get('order_delivered_customer_date')}', Status: '{ped.get('order_status')}'")


--- Iniciando o Pipeline de Sanitização de Dados ---

Total de linhas de produtos processadas: 32951
Categorias de produtos nulas corrigidas para 'Sem Categoria': 610
Dimensões físicas de produtos nulas/vazias corrigidas para '0': 8

Total de linhas de pedidos processadas: 99441
Pedidos com 'order_delivered_customer_date' nula/vazia: 2965
Pedidos cancelados com 'order_delivered_customer_date' nula/vazia: 619
**Hipótese de Negócio Olist PARCIALMENTE COMPROVADA/NÃO COMPROVADA:** Nem todos os pedidos com data de entrega nula têm o status 'canceled'.
    Pedidos com data de entrega nula mas status diferente de 'canceled': 2346
Pedidos com status 'canceled' mas com 'order_delivered_customer_date' preenchida: 6

--- Sumário Estatístico Final ---
Total de linhas processadas (Produtos): 32951
Total de linhas processadas (Pedidos): 99441
Total de registros nulos corrigidos (Categorias de Produtos): 610
Total de registros nulos corrigidos (Dimensões Físicas de Produtos): 8
Total de pedidos canc